<div style="box-sizing:border-box; min-height:12.65in; border:5px solid black; border-radius:6px; padding:48px; display:flex; flex-direction:column; justify-content:center; text-align:center;">
<h1>Milestone 2: Initial Design and Prompt Experiments for Fraudulent Job Posting Detection and Explanation</h1>
<h2>Author: Tim Hollis</h2>
<h2>Bellevue University</h2>
<h2>Course: DSC670 - Advanced Uses of Generative AI</h2>
<h2>Professor Neugebauer</h2>
<h2>Date: July 4, 2026</h2>
</div>

## **Milestone 2: Initial Design and Prompt Experiments**

### **Updates to the Problem Statement**

<p style="text-indent:0.5in; line-height:2;">The core problem from Milestone 1 stands: job seekers facing a suspicious posting get no help from fraud detection research that outputs a label and stops, so this project builds a tool that returns both a verdict and a plain-language explanation of the specific red flags behind it. Two refinements emerged from the first half of this course. First, the dual output is now formalized as a single generation task rather than two loosely coupled goals. Given raw posting text, the model must return one structured response containing a verdict line, a confidence level, and a red flags section written for a non-technical reader. Treating this as one task matters because the experiments below show the verdict and the explanation can disagree in quality, and a fluent rationale attached to a wrong verdict is more dangerous to a job seeker than no tool at all.</p>

<p style="text-indent:0.5in; line-height:2;">Second, the role of fine-tuning has been reframed. Milestone 1 assumed fine-tuning would primarily teach the model the task. The prompt experiments in this milestone show that gpt-4o already performs the task at a moderate level with no examples whatsoever, and that output format is fully controllable through few-shot prompting alone. What prompting could not deliver, across five distinct strategies, was calibrated judgment: no prompt produced a model suspicious enough to catch fraud disguised as a professional posting while remaining fair to legitimate postings with unusual features. Fine-tuning is therefore no longer positioned as capability-building but as judgment-building, which changes what the training data must contain and how the fine-tuned model should be evaluated.</p>

### **Type of Generative AI**

<p style="text-indent:0.5in; line-height:2;">This solution uses a large language model, a text-to-text transformer accessed through the OpenAI API, with gpt-4o serving as the experimentation model in this milestone and gpt-4o-mini as the intended fine-tuning target. The choice follows from the shape of the task. Both the input, a job posting, and the outputs, a verdict and an explanation, are natural language, and the explanation half of the project demands fluent, audience-aware text generation that only an LLM among current generative model families provides. Other generative model types encountered in this course were considered and set aside. Image generation models such as those I compared formally in Assessment 3.2 have no role in a text classification and explanation pipeline, and while a traditional discriminative classifier could handle the verdict alone, as my DSC630 logistic regression model did, it cannot generate the plain-language rationale that defines this project's contribution. A fine-tuned smaller model is preferred over a prompted frontier model for the final application because the task is narrow, the output format is fixed, and inference cost matters for a tool intended to be freely available to job seekers under financial pressure (OpenAI, n.d.).</p>

### **Prompt Experiments**

<p style="text-indent:0.5in; line-height:2;">Five prompt experiments tested whether gpt-4o has the potential to solve this problem before any fine-tuning investment. All five ran against the same test set of five real EMSCAD postings: two confirmed frauds from the work-from-home scam family, two clearly legitimate postings, and one hand-picked borderline case, a commission-based sales posting with an aggressive all-caps title and a $200K earnings claim that EMSCAD nonetheless labels legitimate. Temperature was set to zero throughout so that differences in output reflect prompting strategy rather than sampling randomness. The complete code, model responses, and my full commentary appear in the appendix notebook; this section reproduces each prompt verbatim and summarizes what it revealed.</p>

<p style="text-indent:0.5in; line-height:2;">Experiment 1 established the zero-shot baseline with a minimal one-word classification instruction.</p>

> Classify the following job posting as either FRAUDULENT or LEGITIMATE. Respond with exactly one word.

<p style="text-indent:0.5in; line-height:2;">The model scored three of five, and both misses were informative. It called a real fraud legitimate, the most dangerous error a job seeker protection tool can make, while flagging the borderline sales posting as fraudulent. Notably, it caught the second fraud, which comes from the same scam family as the one it missed, so zero-shot judgment is inconsistent even across similar inputs.</p>

<p style="text-indent:0.5in; line-height:2;">Experiment 2 tested the project's second output by asking for a verdict plus reasoning.</p>

> Classify the following job posting as FRAUDULENT or LEGITIMATE, then explain your reasoning in plain language a non-technical job seeker could understand.

<p style="text-indent:0.5in; line-height:2;">No verdicts changed, but the visible reasoning exposed why the model fails. On the missed fraud, it cited the posting's detailed duties and benefits package as evidence of legitimacy, exactly the features a well-crafted scam imitates. On the borderline posting, it produced a fluent, well-organized explanation naming seven plausible red flags in support of a wrong verdict, the fluent-but-incorrect failure I flagged as a risk in Milestone 1, now observed directly. The five responses also shared no consistent structure, which no application could reliably parse.</p>

<p style="text-indent:0.5in; line-height:2;">Experiment 3 addressed the format problem with few-shot prompting, providing two worked examples, one fraudulent and one legitimate, each answered in the exact structure the final application needs. The system prompt follows; the two full exemplar postings and answers appear in the appendix.</p>

> You classify job postings as FRAUDULENT or LEGITIMATE and explain the red flags in plain language for a non-technical job seeker. Always respond in the exact format shown in the examples.

<p style="text-indent:0.5in; line-height:2;">The result cleanly split the project's two challenges. Every response now opened with a VERDICT line and followed the RED FLAGS structure, so output format proved fully solvable with examples alone. Accuracy did not move: the same three of five, the same two misses, with the polished fraud again judged legitimate on nearly identical reasoning. Two generic exemplars taught format but not judgment.</p>

<p style="text-indent:0.5in; line-height:2;">Experiment 4 tested whether forced step-by-step reasoning could reach the judgment that examples could not, instructing the model to work through a five-part analysis before any verdict.</p>

> You are analyzing a job posting for fraud. Before giving any verdict, work through these steps in order:
>
> 1. Company identity: Is the employer named and verifiable?
> 2. Compensation: Are the pay claims realistic for the role and experience level?
> 3. Contact and application process: Does anything route through personal email or unusual channels?
> 4. Requests of the applicant: Is the applicant asked for money, personal financial details, or unusual commitments?
> 5. Writing quality and pressure tactics: Any urgency, guarantees, or too-good-to-be-true language?
>
> Show your reasoning for each step, then finish with a line in the format VERDICT: FRAUDULENT or VERDICT: LEGITIMATE.

<p style="text-indent:0.5in; line-height:2;">The checklist finally caught the polished fraud, and the transcript shows the mechanism: the mandatory company identity step surfaced that no employer is named, the signal the model's holistic judgment had glossed over in three straight experiments. The cost was a new false positive on a legitimate startup posting, where the same aggressive checklist read placeholder URLs, likely artifacts of the dataset's sanitization, and vague compensation as fraud signals. The score remained three of five, but the error profile transformed to zero false negatives and two false positives, arguably the strongest result yet under this project's recall-first philosophy, since every remaining error is the cheap kind.</p>

<p style="text-indent:0.5in; line-height:2;">Experiment 5 tested role framing as a lightweight preview of fine-tuning, since both aim to make a model behave like a specialist without task instructions in every request. The persona explicitly warned about both failure modes the earlier experiments exposed and added a confidence output.</p>

> You are a fraud analyst with fifteen years of experience protecting job seekers from employment scams. You have reviewed thousands of fraudulent postings and know that sophisticated scams imitate professional job ads, while some legitimate postings, especially aggressive sales roles and early-stage startups, can look suspicious on the surface. Weigh the evidence carefully in both directions before deciding. Respond in this format:
>
> VERDICT: FRAUDULENT or LEGITIMATE
> CONFIDENCE: HIGH, MEDIUM, or LOW
> RED FLAGS: plain-language bullet points a non-technical job seeker can understand, or "None significant" if the posting looks clean

<p style="text-indent:0.5in; line-height:2;">The persona repaired the checklist's false positive, weighing the startup posting's oddities against its detailed funding history and correctly voting legitimate at medium confidence, exactly the calibrated judgment the checklist could not produce. But it lost the polished fraud again, anchoring on professional surface despite an explicit warning that scams imitate professional ads, and it reported high confidence on its remaining error. Across all five experiments the score never moved from three of five, yet no two strategies missed the same way. Prompting redistributed the errors rather than eliminating them. No single prompt gave this model both the suspicion to catch disguised fraud and the fairness to clear unusual but legitimate postings, and that combination, which must be learned from examples at scale, is precisely what fine-tuning provides.</p>

### **Fine-Tuning Approach**

<p style="text-indent:0.5in; line-height:2;">The experiments define the fine-tuning job with unusual precision. Format control is already solved by examples, so the training data's real work is teaching judgment: suspicion calibrated enough to catch polished fraud without overflagging unusual legitimate postings. The method will be supervised fine-tuning of gpt-4o-mini through the OpenAI fine-tuning API (OpenAI, n.d.). Each training example will be a JSONL record in the chat format the API expects: a system message establishing the fraud analyst role, a user message containing the assembled posting text, and an assistant message in the VERDICT, CONFIDENCE, and RED FLAGS structure validated in Experiments 3 and 5. Because EMSCAD provides labels but not explanations (Vidros et al., 2017), the assistant responses must be constructed. My plan is to draft them programmatically from each posting's known fraud indicators, then hand-review a stratified sample for accuracy and readability before training, treating explanation quality as a first-class data quality problem. Experiment 2 showed the model will fluently justify wrong verdicts, so explanations that misstate a posting's contents would train exactly the failure this project exists to prevent.</p>

<p style="text-indent:0.5in; line-height:2;">The training set will deliberately oversample the two failure patterns the experiments exposed: polished frauds that imitate professional postings, and legitimate postings with aggressive or unconventional features such as commission-heavy sales roles and early-stage startups. Evaluation will use a held-out test split never seen during training, scored on recall and F1 for the fraud class, consistent with the recall-first philosophy carried over from my DSC630 work, where missing real fraud costs a job seeker far more than a false alarm costs a second look. Explanation quality will be evaluated separately by reading a sample of generated rationales against the postings they describe and checking that every claimed red flag actually appears in the posting text. The five postings used in this milestone will be retired from evaluation duty, since the model family has now seen them, and will instead serve as the demonstration set for the final Streamlit application.</p>

### **References**

<p style="padding-left:0.5in; text-indent:-0.5in; line-height:2;">OpenAI. (n.d.). <em>Fine-tuning</em>. OpenAI platform documentation. https://platform.openai.com/docs/guides/fine-tuning</p>

<p style="padding-left:0.5in; text-indent:-0.5in; line-height:2;">Vidros, S., Kolias, C., Kambourakis, G., & Akoglu, L. (2017). Automatic detection of online recruitment frauds: Characteristics, methods, and a public dataset. <em>Future Internet, 9</em>(1), 6. https://doi.org/10.3390/fi9010006</p>

<hr style="border: 2px solid black;">

## **Appendix: Prompt Experiment Notebook**

This appendix contains the five prompt experiments supporting Milestone 2: a fine-tuned generative model that classifies job postings as fraudulent or legitimate and explains its reasoning in plain language. Each experiment tests a different prompting strategy against the same set of real postings from the Employment Scam Aegean Dataset (EMSCAD), building a case for whether the problem is solvable with gpt-4o and where fine-tuning will add value. The setup cell below loads all libraries, the EMSCAD dataset, and my OpenAI API key from a local .env file.

## Initial Setup

In [1]:
# Loading Libraries/imports
import os
import textwrap

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown, HTML

display(HTML('''
<style>
    div.output_subarea { page-break-inside: avoid; }
    div.jp-MarkdownOutput { page-break-inside: avoid; }
    div.cell { page-break-inside: avoid; }
</style>
'''))

# Load the API key from .env
load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# Load EMSCAD and confirm the fraud class balance
df = pd.read_csv('fake_job_postings.csv')


def show(text, width=100):
    """Print model output wrapped to a fixed width.
     Args:
        text (str): The model response or any long string to display.
        width (int): Maximum line width before wrapping. Defaults to 100.
    """
    print(textwrap.fill(text, width=width, replace_whitespace=False))


print('✅ Environment ready')
print(f'📊 EMSCAD loaded: {df.shape[0]:,} postings, '
      f'{df["fraudulent"].sum():,} fraudulent')

✅ Environment ready
📊 EMSCAD loaded: 17,880 postings, 866 fraudulent


### **Overview**

The five experiments below follow a deliberate progression. Experiments 1 and 2 establish a zero-shot baseline: can gpt-4o classify a posting cold, and can it explain itself without any guidance on format or content? Experiments 3 and 4 apply the techniques from Weeks 3 and 4 of this course, few-shot prompting with structured exemplars and chain-of-thought reasoning, to see how much control and accuracy improve when the model is shown what good output looks like. Experiment 5 tests a role-based prompt that frames the model as a fraud analyst, which serves as a preview of what fine-tuning should accomplish permanently: a model that behaves like a specialist without being told to in every request.

All five experiments run against the same five postings selected from EMSCAD in the next cell, so differences in output reflect the prompting strategy rather than the input. The selection includes clear fraud, clear legitimate postings, and one harder case, because a tool meant to protect job seekers has to be judged on more than easy examples.

### **Test Posting Selection**

**Step 1:** Filter EMSCAD to postings with enough text to be realistic test cases.

**Step 2:** Sample two fraudulent and two legitimate postings with a fixed random seed so the selection is reproducible.

**Step 3:** Add one hand-picked borderline posting and assemble the final test set with its ground-truth labels hidden from the model.

In [2]:
# Build one shared test set so every experiment sees identical inputs
usable = df[df['description'].str.len() > 300].copy()


def posting_text(row):
    """Assemble one EMSCAD row into the text a job seeker would read.

    Combines the title, location, company profile, description,
    requirements, and benefits fields into a single labeled block,
    skipping any field that is missing so the model never sees a
    literal nan where real content should be.

    Args:
        row (pd.Series): One row of the EMSCAD dataframe.

    Returns:
        str: The formatted posting text used as model input.
    """
    parts = [
        f'Title: {row["title"]}',
        f'Location: {row["location"]}',
        f'Company profile: {row["company_profile"]}',
        f'Description: {row["description"]}',
        f'Requirements: {row["requirements"]}',
        f'Benefits: {row["benefits"]}'
    ]
    return '\n'.join(p for p in parts if 'nan' not in p[:60].lower())


fraud_sample = usable[usable['fraudulent'] == 1].sample(2, random_state=670)
legit_sample = usable[usable['fraudulent'] == 0].sample(2, random_state=670)

# Hand-picked borderline case: aggressive all-caps salary claim on a
# posting EMSCAD labels legitimate, chosen to test false positive behavior
borderline = usable.loc[[78]]

test_set = pd.concat(
    [fraud_sample, legit_sample, borderline]).reset_index(drop=True)
test_set['posting'] = test_set.apply(posting_text, axis=1)

for i, row in test_set.iterrows():
    label = 'FRAUD' if row['fraudulent'] == 1 else 'LEGITIMATE'
    print(f'📋 Posting {i + 1} | Ground truth: {label} | '
          f'Title: {row["title"][:60]}')

print(f'\n✅ Test set assembled: {len(test_set)} postings')

📋 Posting 1 | Ground truth: FRAUD | Title: Customer Service Administrator
📋 Posting 2 | Ground truth: FRAUD | Title: Data Entry Office Support
📋 Posting 3 | Ground truth: LEGITIMATE | Title: Lead Analyst 
📋 Posting 4 | Ground truth: LEGITIMATE | Title: Account Executive - Washington DC
📋 Posting 5 | Ground truth: LEGITIMATE | Title: 200K + MANAGEMENT POSITION FOR EXPERIENCED MERCHANT CASH ADV

✅ Test set assembled: 5 postings


### **Experiment 1: Zero-Shot Classification**

**Step 1:** Send each test posting to gpt-4o with a minimal instruction: classify as fraudulent or legitimate, one word only.

**Step 2:** Compare the model's verdicts against EMSCAD ground truth.

This is the floor. If the model cannot classify postings with no examples and no guidance, the project needs rethinking before any fine-tuning conversation. Temperature is set to 0 across all experiments so differences in output come from the prompt, not sampling randomness.

In [3]:
# Establish the zero-shot baseline every later experiment gets judged against
def run_experiment(system_prompt, user_content, temperature=0):
    """Send one prompt to gpt-4o and return the response text.

    Wraps the chat completions call so each experiment cell stays
    focused on its prompt design rather than API mechanics.

    Args:
        system_prompt (str): The system message framing the task.
        user_content (str): The user message, typically a job posting.
        temperature (float): Sampling temperature. Defaults to 0 so
            results are as deterministic as the API allows.

    Returns:
        str: The model's response text.
    """
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_content}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content


zero_shot_system = (
    'Classify the following job posting as either FRAUDULENT or '
    'LEGITIMATE. Respond with exactly one word.'
)

exp1_results = []
for i, row in test_set.iterrows():
    verdict = run_experiment(zero_shot_system, row['posting'])
    truth = 'FRAUDULENT' if row['fraudulent'] == 1 else 'LEGITIMATE'
    match = '✅' if verdict.strip().upper() == truth else '❌'
    exp1_results.append(verdict)
    print(f'{match} Posting {i + 1} | Truth: {truth} | Model: {verdict}')

print('\n🏁 Experiment 1 complete')

❌ Posting 1 | Truth: FRAUDULENT | Model: LEGITIMATE
✅ Posting 2 | Truth: FRAUDULENT | Model: FRAUDULENT
✅ Posting 3 | Truth: LEGITIMATE | Model: LEGITIMATE
✅ Posting 4 | Truth: LEGITIMATE | Model: LEGITIMATE
❌ Posting 5 | Truth: LEGITIMATE | Model: FRAUDULENT

🏁 Experiment 1 complete


**Summary:** Zero-shot classification got three of five postings right, and the two misses are more informative than the three hits. Posting 1 is a real fraud the model called legitimate, which is the most dangerous error this tool could make: a false negative is exactly the case where a job seeker gets told a scam is safe. Notably, the model caught posting 2, which comes from the same work-from-home scam family, so with no guidance the model is inconsistent even across similar inputs. Posting 5 failed in the opposite direction. The aggressive all-caps title and 200K salary claim read as scam signals, and the model flagged a posting EMSCAD labels legitimate. Under my recall-first design philosophy a false positive is the cheaper mistake, but it still matters, because a tool that cries wolf on every aggressive sales posting will lose the user's trust. The takeaway from this baseline is that gpt-4o clearly has relevant knowledge about job fraud, but with a bare prompt it applies that knowledge unevenly. The remaining experiments test whether prompting technique alone can close that gap.

### **Experiment 2: Zero-Shot with Explanation**

**Step 1:** Ask gpt-4o for a verdict plus a plain-language explanation of its reasoning, with no guidance on format or content.

**Step 2:** Review the explanations for accuracy, readability, and consistency of structure across the five postings.

This experiment tests the second half of my project idea, the explanation, in its most naive form. The question is not just whether the explanations are correct but whether they arrive in a consistent, usable shape without being shown examples.

In [4]:
# Test the dual-output idea with no formatting guidance at all
explain_system = (
    'Classify the following job posting as FRAUDULENT or LEGITIMATE, '
    'then explain your reasoning in plain language a non-technical '
    'job seeker could understand.'
)

exp2_results = []
for i, row in test_set.iterrows():
    answer = run_experiment(explain_system, row['posting'])
    truth = 'FRAUDULENT' if row['fraudulent'] == 1 else 'LEGITIMATE'
    exp2_results.append(answer)
    print(f'📋 Posting {i + 1} | Ground truth: {truth}')
    show(answer)
    print()

print('🏁 Experiment 2 complete')

📋 Posting 1 | Ground truth: FRAUDULENT
The job posting for a Customer Service Administrator appears to be LEGITIMATE. Here's why:

1.
**Specific Job Description**: The posting provides a clear and detailed description of the job
responsibilities, such as processing forms, handling customer interactions, and resolving billing
and shipping errors. This specificity is typical of legitimate job postings.

2. **Clear
Requirements**: The job lists specific requirements, including a minimum of two years of experience,
a high school diploma, and preferred skills in Word, Excel, and MS Office. Legitimate postings often
have clear criteria for applicants.

3. **Industry Mention**: It mentions the Consumer Packaged
Goods industry, which adds credibility as it aligns with a real sector where such roles are common.
4. **Benefits and Compensation**: The posting outlines a comprehensive benefits package, including a
401k with matching and competitive wages. Legitimate jobs often provide details about

**Summary:** Asking for explanations did not change any verdicts. The score is still three of five with the same two misses, but the reasoning text reveals why the model fails, which the one-word experiment could not. On posting 1, the model explicitly cites the posting's detailed duties, clear requirements, and benefits package as evidence of legitimacy. Those are exactly the features a well-crafted scam imitates, so the model's zero-shot heuristic amounts to judging professionalism rather than authenticity, and polished fraud defeats it. Posting 5 is the more unsettling result. The explanation is fluent, well organized, and names seven plausible red flags, and it is wrong. A job seeker reading it would walk away from a posting EMSCAD labels legitimate, fully convinced. This is the fluent-but-incorrect failure I flagged in Milestone 1 as a risk, now observed directly. Finally, the five responses share no consistent structure. Two lead with a one-word verdict, three bury the verdict inside a sentence, and formatting ranges from numbered lists to paragraphs of varying length. An application cannot reliably parse or display output this inconsistent, which is precisely the problem the next experiment addresses with structured examples.

### **Experiment 3: Few-Shot with Structured Examples**

**Step 1:** Build a few-shot prompt containing two worked examples, one fraudulent and one legitimate posting, each answered in the exact output format the final application needs.

**Step 2:** Run all five test postings through the few-shot prompt and check both verdict accuracy and format compliance.

The examples define the target structure: a VERDICT line followed by a RED FLAGS section written for a non-technical reader. This is also a preview of the fine-tuning training data, since each fine-tuning example will pair a posting with a response in this same shape.

In [5]:
# Show the model the exact output shape the application needs
few_shot_system = (
    'You classify job postings as FRAUDULENT or LEGITIMATE and explain '
    'the red flags in plain language for a non-technical job seeker. '
    'Always respond in the exact format shown in the examples.'
)

example_fraud = (
    'Title: Work From Home Typist\n'
    'Description: Earn $500 daily typing from home! No experience needed. '
    'Guaranteed income. Small registration fee required to get started. '
    'Contact us at quickhire99@gmail.com to begin immediately.'
)

example_fraud_answer = (
    'VERDICT: FRAUDULENT\n'
    'RED FLAGS:\n'
    '- Promises high guaranteed pay for unskilled work, which real '
    'employers do not offer\n'
    '- Asks you to pay a registration fee, and legitimate jobs never '
    'require payment to start\n'
    '- Uses a free personal email address instead of a company domain'
)

example_legit = (
    'Title: Staff Accountant\n'
    'Description: Regional accounting firm seeks staff accountant with '
    '2+ years experience. CPA preferred. Responsibilities include monthly '
    'close, reconciliations, and client reporting. Salary $55,000-$65,000 '
    'based on experience. Full benefits. Apply via our careers page.'
)

example_legit_answer = (
    'VERDICT: LEGITIMATE\n'
    'RED FLAGS:\n'
    '- None significant. The posting names a realistic salary range, '
    'specific qualifications, and concrete job duties, and it directs '
    'applicants to a company careers page rather than a personal email'
)

exp3_results = []
for i, row in test_set.iterrows():
    messages = [
        {'role': 'system', 'content': few_shot_system},
        {'role': 'user', 'content': example_fraud},
        {'role': 'assistant', 'content': example_fraud_answer},
        {'role': 'user', 'content': example_legit},
        {'role': 'assistant', 'content': example_legit_answer},
        {'role': 'user', 'content': row['posting']}
    ]
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=messages,
        temperature=0
    )
    answer = response.choices[0].message.content
    truth = 'FRAUDULENT' if row['fraudulent'] == 1 else 'LEGITIMATE'
    exp3_results.append(answer)
    print(f'📋 Posting {i + 1} | Ground truth: {truth}')
    show(answer)
    print()

print('🏁 Experiment 3 complete')

📋 Posting 1 | Ground truth: FRAUDULENT
VERDICT: LEGITIMATE
RED FLAGS:
- None significant. The job description provides specific duties,
realistic requirements, and a detailed benefits package, which are typical of legitimate job
postings.

📋 Posting 2 | Ground truth: FRAUDULENT
VERDICT: FRAUDULENT
RED FLAGS:
- Claims of wages ranging from $12 per hour to a six-figure salary
are unrealistic and overly broad for data entry and similar roles
- Vague job description without
specific company details or job duties
- No clear application process or company contact information
provided
- Promises of bonuses and benefits without specifying the employer or job specifics

📋 Posting 3 | Ground truth: LEGITIMATE
VERDICT: LEGITIMATE

RED FLAGS:
- None significant. The job posting provides detailed information
about the company, its funding, and its growth, which is typical for legitimate job postings. It
also outlines specific job responsibilities, qualifications, and benefits, and uses professional

**Summary:** Few-shot prompting solved exactly one of the two problems identified in Experiment 2. Every response now opens with a VERDICT line and follows the RED FLAGS structure from the exemplars, with only trivial whitespace variation, so the output is finally something an application could parse and display reliably. Accuracy, however, did not move. The score is still three of five with the identical two misses, and the reasoning on posting 1 repeats the same legitimacy heuristic from the previous experiment almost verbatim: detailed duties and a benefits package are read as proof of authenticity. Two generic examples were enough to teach format but not enough to teach judgment, which makes sense, since neither exemplar demonstrated the specific pattern posting 1 represents, a polished scam that imitates professional postings. This result cleanly separates the project's two challenges. Output structure is a prompting problem and is now solved. Classification judgment on sophisticated fraud is a deeper problem that two exemplars cannot fix, and the remaining experiments test whether reasoning technique or role framing can close that gap before concluding that fine-tuning is required.

### **Experiment 4: Chain-of-Thought Reasoning**

**Step 1:** Instruct the model to work through a structured analysis of the posting, examining company identity, compensation claims, contact methods, and requests made of the applicant, before committing to a verdict.

**Step 2:** Check whether deliberate step-by-step reasoning changes any verdicts, especially on posting 1, the polished fraud that every strategy so far has missed.

Chain-of-thought prompting improved arithmetic reasoning in my Week 4 exercises by making the model show its work before answering. The hypothesis here is the same: forcing an explicit feature-by-feature analysis may surface fraud signals that the model's quick holistic judgment glosses over.

In [6]:
# Force a structured analysis before the verdict instead of after it
cot_system = (
    'You are analyzing a job posting for fraud. Before giving any '
    'verdict, work through these steps in order:\n'
    '1. Company identity: Is the employer named and verifiable?\n'
    '2. Compensation: Are the pay claims realistic for the role and '
    'experience level?\n'
    '3. Contact and application process: Does anything route through '
    'personal email or unusual channels?\n'
    '4. Requests of the applicant: Is the applicant asked for money, '
    'personal financial details, or unusual commitments?\n'
    '5. Writing quality and pressure tactics: Any urgency, guarantees, '
    'or too-good-to-be-true language?\n'
    'Show your reasoning for each step, then finish with a line in the '
    'format VERDICT: FRAUDULENT or VERDICT: LEGITIMATE.'
)

exp4_results = []
for i, row in test_set.iterrows():
    answer = run_experiment(cot_system, row['posting'])
    truth = 'FRAUDULENT' if row['fraudulent'] == 1 else 'LEGITIMATE'
    exp4_results.append(answer)
    print(f'📋 Posting {i + 1} | Ground truth: {truth}')
    show(answer)
    print()

print('🏁 Experiment 4 complete')

📋 Posting 1 | Ground truth: FRAUDULENT
1. Company identity: The job posting does not mention the name of the employer, which makes it
difficult to verify the legitimacy of the company. A legitimate job posting typically includes the
company's name to allow potential applicants to research and verify its authenticity.

2.
Compensation: The job posting mentions "competitive wages, based on education and experience," but
does not provide specific salary information. While this is not uncommon, it does not provide enough
information to assess whether the pay claims are realistic.

3. Contact and application process: The
job posting does not provide any details about how to apply or contact the employer. There is no
mention of a company website, official email address, or application portal, which is unusual for a
legitimate job posting.

4. Requests of the applicant: The job posting does not ask for money,
personal financial details, or unusual commitments from the applicant. However, the 

**Summary:** Chain-of-thought reasoning finally caught posting 1, and the transcript shows exactly why. Step 1 of the checklist forced the model to ask whether the employer is named and verifiable, and it is not, which was the fraud signal hiding under the posting's professional surface the whole time. Three previous experiments let the model judge holistically and it anchored on polish; a mandatory feature-by-feature analysis surfaced what polish was covering. Both true frauds are now caught, and posting 5 remains flagged with a detailed rationale. The catch is posting 3, a legitimate startup posting that flipped to a false positive under the same aggressive checklist. The model cited placeholder URLs, which are likely artifacts of the dataset's sanitization rather than the original posting, along with vague compensation and a missing application process. The overall score is unchanged at three of five, but the error profile transformed: zero false negatives and two false positives. Under my recall-first philosophy this is the strongest result so far, since every miss is now the cheap kind that costs a second look rather than a job seeker's savings. What it also shows is that prompting can tune the model's suspicion up or down but has not yet delivered accuracy and calibration together. That is the gap the final experiment probes with role framing, and if it persists, it defines precisely what fine-tuning needs to fix.

### **Experiment 5: Role-Based Prompt**

**Step 1:** Frame the model as an experienced fraud analyst who protects job seekers, combining a persona with the output format from Experiment 3, and let it weigh the evidence rather than march through a fixed checklist.

**Step 2:** Compare verdicts and calibration against the previous four experiments, watching whether the persona balances the recall gains of Experiment 4 against its false positive cost.

A persona prompt is a lightweight preview of fine-tuning: both aim to make the model behave like a specialist without task instructions in every request. If role framing achieves calibrated judgment here, fine-tuning's job is mainly consistency and format. If it does not, fine-tuning also carries the burden of teaching judgment.

In [7]:
# Test whether a specialist persona delivers judgment that checklists could not
persona_system = (
    'You are a fraud analyst with fifteen years of experience protecting '
    'job seekers from employment scams. You have reviewed thousands of '
    'fraudulent postings and know that sophisticated scams imitate '
    'professional job ads, while some legitimate postings, especially '
    'aggressive sales roles and early-stage startups, can look '
    'suspicious on the surface. Weigh the evidence carefully in both '
    'directions before deciding. Respond in this format:\n'
    'VERDICT: FRAUDULENT or LEGITIMATE\n'
    'CONFIDENCE: HIGH, MEDIUM, or LOW\n'
    'RED FLAGS: plain-language bullet points a non-technical job seeker '
    'can understand, or "None significant" if the posting looks clean'
)

exp5_results = []
for i, row in test_set.iterrows():
    answer = run_experiment(persona_system, row['posting'])
    truth = 'FRAUDULENT' if row['fraudulent'] == 1 else 'LEGITIMATE'
    exp5_results.append(answer)
    print(f'📋 Posting {i + 1} | Ground truth: {truth}')
    show(answer)
    print()

print('🏁 Experiment 5 complete')

📋 Posting 1 | Ground truth: FRAUDULENT
VERDICT: LEGITIMATE  
CONFIDENCE: MEDIUM  
RED FLAGS:  
- None significant. The job description and
requirements are typical for a Customer Service Administrator role.  
- The benefits package is
standard and includes details like a 401k with matching, which is a positive sign.  
- The posting
lacks specific company information, which is not uncommon but could be improved for transparency.
Overall, the job posting appears to be legitimate, but job seekers should verify the company's
details and ensure communication is through official channels.

📋 Posting 2 | Ground truth: FRAUDULENT
VERDICT: FRAUDULENT  
CONFIDENCE: HIGH  
RED FLAGS:
- Vague job description that lacks specific
details about the company or the exact nature of the work.
- Unrealistic salary claims, such as a
range from $12 per hour to a six-figure salary, which is highly unusual for data entry positions.
-
No company name or contact information provided, making it difficult to veri

**Summary:** The persona experiment completes a revealing pattern. It repaired the false positive from Experiment 4, weighing posting 3's placeholder URLs and generic company name against its detailed funding history and correctly voting legitimate at medium confidence, which is exactly the calibrated judgment the checklist could not produce. But it lost posting 1 again. Despite a system prompt that explicitly warned that sophisticated scams imitate professional job ads, the model anchored on the professional surface one more time. Across all five experiments the score never moved from three of five, yet no two strategies missed the same way. The checklist alone caught the polished fraud but overflagged a scrappy startup; judgment-based prompts calibrated on the startup but fell for the polished fraud; and posting 5 was flagged by everything, at high confidence here, though I would argue EMSCAD's legitimate label on that posting is itself debatable given its multi-level marketing structure. The new confidence signal was also miscalibrated where it mattered most, reporting high confidence on the posting 5 error. The conclusion I draw is that prompting redistributes errors rather than eliminating them. No single prompt gave this model both the suspicion to catch disguised fraud and the judgment to clear unusual-but-legitimate postings. That combination has to be learned from examples at scale, which is precisely what fine-tuning provides and what the next section of this milestone proposes.

### **Fine-Tuning Plan**

The experiments above define the fine-tuning job precisely. Format control is already solved by examples, so the training data's main work is teaching judgment: suspicion calibrated enough to catch polished fraud without overflagging unusual legitimate postings.

**Training data construction.** Each training example will be a JSONL record in the chat format OpenAI's fine-tuning API expects: a system message establishing the fraud analyst role, a user message containing the assembled posting text, and an assistant message in the VERDICT / CONFIDENCE / RED FLAGS format from Experiments 3 and 5. Because EMSCAD provides labels but not explanations, the assistant responses must be constructed. My plan is to draft them programmatically from each posting's known fraud indicators, then hand-review a stratified sample for accuracy and readability before training, treating explanation quality as a first-class data quality problem rather than an afterthought. The training set will deliberately oversample the two failure patterns these experiments exposed: polished frauds that imitate professional postings, and legitimate postings with aggressive or unconventional features such as commission-heavy sales roles and early-stage startups.

**Model and method.** Supervised fine-tuning of gpt-4o-mini through the OpenAI fine-tuning API. The task is narrow and the target format is fixed, which is the profile where a smaller fine-tuned model typically matches or beats a larger prompted one at a fraction of the inference cost, an important consideration for a tool meant to be freely usable by job seekers.

**Evaluation.** A held-out test split, never seen during training, scored on recall and F1 for the fraud class, consistent with the recall-first philosophy carried over from my DSC630 work. Explanation quality will be evaluated separately by reading a sample of generated rationales against the postings they describe, checking that every claimed red flag actually appears in the posting, since Experiment 2 demonstrated the model can produce fluent explanations for wrong verdicts. The five-posting test set from this notebook will be retired into the demonstration set for the final Streamlit app, since it can no longer serve as unseen data.

### **Summary**

This notebook ran five prompt experiments against a shared test set of five real EMSCAD postings to determine whether gpt-4o can support a tool that classifies job postings and explains its reasoning to non-technical job seekers. The answer is a qualified yes. The model clearly holds relevant knowledge about employment fraud, and few-shot examples fully solved the output format problem, producing parseable VERDICT and RED FLAGS responses on every posting. What no prompting strategy solved is calibrated judgment. All five experiments scored three of five, but each missed differently: holistic prompts fell for a polished fraud that imitates professional postings, an aggressive checklist caught that fraud but overflagged a legitimate startup, and a persona prompt restored calibration on the startup while losing the polished fraud again. Prompting redistributed the errors without eliminating them, and the model's confidence signal was highest on one of its wrong answers. These results justify the project's central design decision: fine-tuning gpt-4o-mini on constructed examples that pair EMSCAD postings with verdicts and vetted plain-language explanations, oversampling exactly the failure patterns these experiments surfaced. The prompt experiments did their job, proving the problem is within reach of this model family while mapping precisely where the remaining gap lies.

<hr style="border: 2px solid black;">